In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
import matplotlib.pyplot as plt
import seaborn as sns

import os

os.environ["USE_TF"] = "0"
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, classification_report

/home/pierre/miniconda3/envs/ia/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../../data/MovieDataThread.csv")

In [3]:
df = df[df["imdb_id"].notna()]
df.drop(columns="imdb_id", inplace=True)
df.drop(columns=df.filter(like="tmdb_").columns, inplace=True)

In [4]:
df["genre_count"] = df.filter(regex='^imdb_(?!id$)').count(axis=1)

In [5]:
df = df[df["genre_count"] > 0]


---
# Multi label

In [6]:
from sklearn.preprocessing import MultiLabelBinarizer

In [40]:
genre_counts = df.drop(columns="imdb_drama").filter(like='imdb_').count().sort_values(ascending=False)
# first quartile of genre counts
genre_counts_quartiles = genre_counts.quantile([0.25, 0.5, 0.75])
# Select genres that are in the top quartile
top_genres = genre_counts[genre_counts > genre_counts_quartiles[0.75]].index
top_genres = top_genres.sort_values(key=lambda x: genre_counts[x], ascending=False)
# Filter the movies to include only thoses with all genres in the top quartile
df_most_genres = df[df[top_genres].notna().sum(axis=1) == df["genre_count"]]

In [41]:
X = df_most_genres['Script']
y = []
for _, row in df_most_genres.iterrows():
    genres = [col for col in df_most_genres.columns if col.startswith('imdb_') and not pd.isna(row[col])]
    y.append(genres)

In [42]:
# Tronquer le texte
X = X.apply(lambda x: ' '.join(str(x).split()[:512]))

# Encoder les labels en vecteurs binaires
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(y)


train_texts, test_texts, train_labels, test_labels = train_test_split(X, y_encoded, test_size=0.2, random_state=42)


train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'labels': train_labels.astype("float32").tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'labels': test_labels.astype("float32").tolist()})

model_checkpoint = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])


train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(mlb.classes_),
    problem_type="multi_label_classification"
)

# Métriques multi-label
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = (logits > 0).astype(int)
    return {
        'precision': precision_score(labels, preds, average='weighted', zero_division=0),
        'recall': recall_score(labels, preds, average='weighted', zero_division=0),
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted', zero_division=0)
    }


training_args = TrainingArguments(
    output_dir="./results_multi_label",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


trainer.train()


predictions = trainer.predict(test_dataset)
y_preds = (predictions.predictions > 0).astype(int)


print(classification_report(test_labels, y_preds, target_names=mlb.classes_))

Map: 100%|██████████| 1602/1602 [00:00<00:00, 3273.16 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-MiniLM-L6-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/pierre/miniconda3/envs/ia/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch,Training Loss,Validation Loss,Precision,Recall,Accuracy,F1
1,0.483900,0.459711,0.580693,0.216832,0.136080,0.280841
2,0.436600,0.424856,0.633806,0.456082,0.273408,0.507712
3,0.414200,0.416489,0.610707,0.458287,0.267166,0.497827
4,0.390500,0.414878,0.627386,0.503859,0.294632,0.542959
5,0.364100,0.416952,0.618200,0.505696,0.274032,0.538661
6,0.349500,0.425876,0.612025,0.526277,0.277778,0.549349


                precision    recall  f1-score   support

   imdb_action       0.69      0.48      0.57       358
imdb_adventure       0.00      0.00      0.00       102
   imdb_comedy       0.74      0.73      0.73       776
    imdb_crime       0.53      0.06      0.11       249
   imdb_horror       0.68      0.63      0.66       434
  imdb_romance       0.55      0.40      0.46       279
 imdb_thriller       0.58      0.44      0.50       523

     micro avg       0.67      0.50      0.57      2721
     macro avg       0.54      0.39      0.43      2721
  weighted avg       0.63      0.50      0.54      2721
   samples avg       0.63      0.56      0.56      2721



/home/pierre/miniconda3/envs/ia/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/pierre/miniconda3/envs/ia/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [45]:
from sklearn.metrics import accuracy_score

print("Exact Match Accuracy:", accuracy_score(test_labels, y_preds))


Exact Match Accuracy: 0.29463171036204744


In [46]:
model.save_pretrained("./results_multi_label")
tokenizer.save_pretrained("./results_multi_label")

('./results_multi_label/tokenizer_config.json',
 './results_multi_label/special_tokens_map.json',
 './results_multi_label/vocab.txt',
 './results_multi_label/added_tokens.json',
 './results_multi_label/tokenizer.json')

---
## Without drama

In [30]:
genre_counts = df.drop(columns="imdb_drama").filter(like='imdb_').count().sort_values(ascending=False)
# first quartile of genre counts
genre_counts_quartiles = genre_counts.quantile([0.25, 0.5, 0.75])
# Select genres that are in the top quartile
top_genres = genre_counts[genre_counts > genre_counts_quartiles[0.75]].index
top_genres = top_genres.sort_values(key=lambda x: genre_counts[x], ascending=False)
# Filter the movies to include only thoses with all genres in the top quartile
df_most_genres = df[df[top_genres].notna().sum(axis=1) == df["genre_count"]]

In [31]:
X = df_most_genres['Script']
y = []
for _, row in df_most_genres.iterrows():
    genres = [col for col in df_most_genres.columns if col.startswith('imdb_') and not pd.isna(row[col])]
    y.append(genres)

In [32]:
top_genres

Index(['imdb_comedy', 'imdb_thriller', 'imdb_romance', 'imdb_action',
       'imdb_horror', 'imdb_crime', 'imdb_adventure'],
      dtype='object')

In [33]:
# Tronquer le texte
X = X.apply(lambda x: ' '.join(str(x).split()[:512]))

# Encoder les labels en vecteurs binaires
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(y)


train_texts, test_texts, train_labels, test_labels = train_test_split(X, y_encoded, test_size=0.2, random_state=42)


train_dataset = Dataset.from_dict({'text': train_texts.tolist(), 'labels': train_labels.astype("float32").tolist()})
test_dataset = Dataset.from_dict({'text': test_texts.tolist(), 'labels': test_labels.astype("float32").tolist()})

model_checkpoint = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])


train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(mlb.classes_),
    problem_type="multi_label_classification"
)

# Métriques multi-label
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = (logits > 0).astype(int)
    return {
        'precision': precision_score(labels, preds, average='weighted', zero_division=0),
        'recall': recall_score(labels, preds, average='weighted', zero_division=0),
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted', zero_division=0)
    }


training_args = TrainingArguments(
    output_dir="./results_multi_label_without_drama",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


trainer.train()


predictions = trainer.predict(test_dataset)
y_preds = (predictions.predictions > 0).astype(int)


print(classification_report(test_labels, y_preds, target_names=mlb.classes_))

Map: 100%|██████████| 1602/1602 [00:00<00:00, 3186.89 examples/s]
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/all-MiniLM-L6-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/pierre/miniconda3/envs/ia/lib/python3.9/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Epoch,Training Loss,Validation Loss,Precision,Recall,Accuracy,F1
1,0.480300,0.458837,0.507290,0.248806,0.141698,0.315663
2,0.435800,0.426441,0.584535,0.418229,0.274032,0.476150
3,0.416800,0.418494,0.588068,0.453510,0.252809,0.487928
4,0.394700,0.419471,0.617938,0.496509,0.296504,0.530687
5,0.363000,0.418932,0.617352,0.502389,0.281523,0.538010
6,0.349600,0.422667,0.603910,0.528482,0.282147,0.550676


                precision    recall  f1-score   support

   imdb_action       0.71      0.46      0.56       358
imdb_adventure       0.00      0.00      0.00       102
   imdb_comedy       0.72      0.72      0.72       776
    imdb_crime       0.50      0.05      0.09       249
   imdb_horror       0.64      0.69      0.67       434
  imdb_romance       0.60      0.34      0.43       279
 imdb_thriller       0.57      0.42      0.49       523

     micro avg       0.66      0.50      0.57      2721
     macro avg       0.53      0.38      0.42      2721
  weighted avg       0.62      0.50      0.53      2721
   samples avg       0.63      0.56      0.56      2721



/home/pierre/miniconda3/envs/ia/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/pierre/miniconda3/envs/ia/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [34]:
from sklearn.metrics import accuracy_score

print("Exact Match Accuracy:", accuracy_score(test_labels, y_preds))


Exact Match Accuracy: 0.2965043695380774


In [35]:
model.save_pretrained("./results_multi_label_without_drama")
tokenizer.save_pretrained("./results_multi_label_without_drama")

('./results_multi_label_without_drama/tokenizer_config.json',
 './results_multi_label_without_drama/special_tokens_map.json',
 './results_multi_label_without_drama/vocab.txt',
 './results_multi_label_without_drama/added_tokens.json',
 './results_multi_label_without_drama/tokenizer.json')

In [37]:
df_most_genres

,Script,Title,imdb_action,imdb_adventure,imdb_animation,imdb_biography,imdb_comedy,imdb_crime,imdb_documentary,imdb_drama,...,imdb_sci-fi,imdb_short,imdb_sport,imdb_talk-show,imdb_thriller,imdb_war,imdb_western,synopsis,imdb_adult,genre_count
0,So what's goin' on?\n Where's my brother?\n So...,Beyond the Dunwich Horror (2008),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
11,- I hope those lines hold.\n - Hold on. Hold o...,Beyond the Poseidon Adventure (1979),1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The film continues the story from the original...,NaN,2
12,"You know I can't keep this, right?\n Your gran...",Beyond the Reach (2014),1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,3
20,"Arjun, where are you...???\n Arjun, what happe...",Bhaag Saale (2023),NaN,NaN,NaN,NaN,1.0,2.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
21,CBI special director Mr. Dhanunjayis waiting f...,Bhaagamathie (2018),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38139,1\n A WONDERFUL LOVE STORY\n Okay. We're going...,[REC] 3: Genesis (2012),1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3.0,NaN,NaN,In '[REC] 3: Genesis' the action now takes pla...,NaN,3
38140,1\n No detailed information has been received\...,[REC] 4: Apocalypse (2014),1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,3
38152,"1\n If someone asks me,\n ""What is the most un...",iGirl (2016),NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
38156,1\n Spotter one in position.\n Hillbrow spotte...,iNumber Number: Jozi Gold (2023),1.0,2.0,NaN,NaN,NaN,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3


---
# Test Multi-label

In [47]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import pandas as pd

# Charger le modèle et le tokenizer
model_checkpoint = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained("./results_multi_label")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Simule ton encoder (ou charge-le si tu l'as sauvegardé)
# Extract all genres from columns starting with 'imdb_' in df_most_genres
all_genres = [col for col in top_genres]
mlb = MultiLabelBinarizer(classes=all_genres)
mlb.fit([all_genres])  # pour garder l'ordre

def predict_genres(phrase, threshold=0.5):
    # Tokenisation
    inputs = tokenizer(phrase, return_tensors="pt", truncation=True, padding=True, max_length=512)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.sigmoid(logits)  # Multi-label => sigmoid
        predictions = (probs > threshold).int().cpu().numpy()

    # predictions est déjà (1, n_classes)
    predicted_labels = mlb.inverse_transform(predictions)[0]

    return predicted_labels


# Test
test_phrase = "Late at night, a strange figure lurked in the shadows, its eyes glowing red as it slowly approached the terrified victim"
predicted_genres = predict_genres(test_phrase, threshold=0.4)
print(f"Genres prédits : {predicted_genres}")


Genres prédits : ('imdb_horror', 'imdb_adventure')
